## Load useful libraries

In [16]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cosine

In [2]:
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
import pyspark.sql.functions as F
from pyspark.sql import Window
from pyspark.sql.types import FloatType

## User settings

In [25]:
output_directory = 'output'
sampling_rate = 22050
hop_length = 512 * 100
spark_memory = '70G'
percentile_cutoff = 0.8
approx_quantile_precision = 0.05
n_to_keep = 2

# location of tracks to analyze for building the track feature database
path_library_parquet = '../database/music/output/playlist.parquet'

## Initialize Spark session

In [4]:
conf = (
    SparkConf()
    .setAppName('AnalyzedTrackLibrary')
    .set('spark.executor.memory', spark_memory)
    .set('spark.driver.memory', spark_memory)
    .set('spark.driver.maxResultSize', spark_memory)
)

spark = SparkSession.builder.config(conf = conf).getOrCreate()

26/03/19 17:41:59 WARN Utils: Your hostname, emily-MS-7B96 resolves to a loopback address: 127.0.1.1; using 192.168.1.99 instead (on interface eno1)
26/03/19 17:41:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/19 17:41:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Load show and track library data

In [7]:
path_show_output = output_directory + '/show_vectors_spark_sr_' + str(sampling_rate) + '_hl_' + str(hop_length) + '.parquet'
sdf_show = spark.read.parquet(path_show_output)

In [8]:
path_library_output = output_directory + '/library_vectors_spark_sr_' + str(sampling_rate) + '_hl_' + str(hop_length) + '.parquet'
sdf_library = spark.read.parquet(path_library_output)

## Join the show and track library dataframes

In [20]:
sdf_cross_joined = (
    sdf_show
    .crossJoin(sdf_library)
    .orderBy('time_step', 'id')
)

In [21]:
sdf_cross_joined.show(3)

+---------+--------------------+--------------------+---+
|time_step|          array_show|       array_library| id|
+---------+--------------------+--------------------+---+
|        0|[0.29534608125686...|[0.28707078099250...| 28|
|        0|[0.29534608125686...|[0.29539421200752...| 28|
|        0|[0.29534608125686...|[0.28956532478332...| 28|
+---------+--------------------+--------------------+---+
only showing top 3 rows



## Define a function for computing cosine similarity

In [22]:
@F.udf(returnType=FloatType())
def compute_cosine_similarity(vector1, vector2):
    cosine_dist = cosine(np.array(vector1), np.array(vector2))
    similarity_score = 1 - cosine_dist
    return float(similarity_score)

## Compute cosine similarity

In [23]:
sdf_cross_joined = (
    sdf_cross_joined
    .withColumn('cosine_similarity', compute_cosine_similarity(F.col('array_show'), F.col('array_library')))
    .drop('array_show', 'array_library')
    .orderBy('time_step', 'id')
)

In [24]:
sdf_cross_joined.show(5)

+---------+---+-----------------+
|time_step| id|cosine_similarity|
+---------+---+-----------------+
|        0| 28|       0.68394727|
|        0| 28|        0.6860955|
|        0| 28|          0.68217|
|        0| 28|        0.6945333|
|        0| 28|        0.6886212|
+---------+---+-----------------+
only showing top 5 rows



## Reduce dataset size by percentile cutoff

In [26]:
sdf_cross_joined.repartition(100)  # I just made this number up, there is no specific rationale for it.

DataFrame[time_step: bigint, id: bigint, cosine_similarity: float]

In [27]:
sdf_cross_joined.count()

146762760

In [28]:
similarity_quantile_cutoff = sdf_cross_joined.approxQuantile('cosine_similarity', [percentile_cutoff], approx_quantile_precision)

In [29]:
similarity_quantile_cutoff

[0.8558716177940369]

In [30]:
sdf_cross_joined = (
    sdf_cross_joined
    .where(F.col('cosine_similarity') >= F.lit(similarity_quantile_cutoff[0]))
    .orderBy('time_step', 'id')
)

In [ ]:
sdf_cross_joined.count()

26/03/19 18:17:23 WARN ExtractPythonUDFFromJoinCondition: The join condition:(compute_cosine_similarity(array_show#5, array_library#8)#105 >= 0.8558716) of the join plan contains PythonUDF only, it will be moved out and the join plan will be turned to cross join.


## Aggregate by (timestamp, song_id)

We retain the maximum cosine similarity per (timestamp / song ID) pair:

In [ ]:
sdf_cross_joined.repartition('time_step', 'id')

sdf_agg = (
    sdf_cross_joined
    .groupBy('timestamp', 'id')
    .agg(
        F.max('cosine_similarity').alias('cosine_similarity'),
    )
    .orderBy('time_step', F.desc('cosine_similarity'))
)

## Record the rank per timestamp¶

In [ ]:
sdf_agg.repartition('timestamp')

In [ ]:
window_spec = Window.partitionBy('timestamp').orderBy(F.desc('cosine_similarity'))

sdf_agg_ranked = (
    sdf_agg
    .orderBy(F.asc('timestamp'), F.desc('cosine_similarity'))
    .withColumn('rank', F.row_number().over(window_spec))
)

In [ ]:
sdf_agg_ranked.show(5)

## Keep only the top-ranked rows per time step

In [ ]:
sdf_agg_ranked = sdf_agg_ranked.where(F.col('rank') <= n_to_keep)

## Load the track titles and artists

In [ ]:
sdf_library_names = (
    spark
    .createDataFrame(pd.read_parquet(path_library_parquet))
    .drop('path')
    .orderBy('id')
)

In [ ]:
sdf_library_names.show(3)